# Lab 3: Propositional Logic and First-Order Logic in Python

## **Muhammad Sikandar Hussain 502808** 

In [1]:
# itertools.product will generate every possible truth-value assignment for us.
import itertools

## Task 1 – Propositions as Python Booleans

A proposition is a declarative sentence that is either true or false. Commands and questions are not propositions because they do not claim something with a truth value.

In [2]:
# Proposition: this factual statement is true.
islamabad_is_capital = True

# Not a proposition: 'Please submit your assignment.' is a command.
# Not a proposition: 'Is the network connected?' is a question.

# Proposition: its real-world truth depends on the server's current condition.
# I assume the server is down for this example.
server_down = True

# Proposition: this mathematical comparison is false.
ten_is_greater_than_twenty = 10 > 20

# The conditional statement is also a proposition. Its two smaller propositions
# are represented separately; this example assumes the website is unavailable.
website_unavailable = True

def website_status(server_is_down, site_is_unavailable):
    """Return the truth value of: server down -> website unavailable."""
    return (not server_is_down) or site_is_unavailable

print("Islamabad is Pakistan's capital:", islamabad_is_capital)
print("10 > 20:", ten_is_greater_than_twenty)
print("Server implication:", website_status(server_down, website_unavailable))

# A direct check of Modus Ponens from the introductory example.
if website_status(server_down, website_unavailable) and server_down:
    print("Modus Ponens conclusion - website unavailable:", website_unavailable)

Islamabad is Pakistan's capital: True
10 > 20: False
Server implication: True
Modus Ponens conclusion - website unavailable: True


## Task 2 – Truth Tables with `itertools.product`

In [3]:
# Part A: helper functions for the five propositional connectives.
def neg(P):
    return not P

def conj(P, Q):
    return P and Q

def disj(P, Q):
    return P or Q

def impl(P, Q):
    return (not P) or Q

def bicond(P, Q):
    return P == Q

print(f"{'P':<6}{'Q':<6}{'~P':<6}{'P&Q':<6}{'P|Q':<6}{'P->Q':<7}{'P<->Q':<6}")
for P, Q in itertools.product([True, False], repeat=2):
    print(f"{str(P):<6}{str(Q):<6}{str(neg(P)):<6}{str(conj(P,Q)):<6}"
          f"{str(disj(P,Q)):<6}{str(impl(P,Q)):<7}{str(bicond(P,Q)):<6}")

P     Q     ~P    P&Q   P|Q   P->Q  P<->Q
True  True  False True  True  True   True  
True  False False False True  False  False 
False True  True  False True  True   False 
False False True  False False True   True  


In [4]:
# Part B: truth table for (P OR Q) -> NOT R.
def compound(P, Q, R):
    return impl(disj(P, Q), neg(R))

print(f"{'P':<6}{'Q':<6}{'R':<6}(P or Q) -> not R")
for P, Q, R in itertools.product([True, False], repeat=3):
    print(f"{str(P):<6}{str(Q):<6}{str(R):<6}{compound(P, Q, R)}")

P     Q     R     (P or Q) -> not R
True  True  True  False
True  True  False True
True  False True  False
True  False False True
False True  True  False
False True  False True
False False True  True
False False False True


## Task 3 – Satisfiability and Validity Checker

A valid formula is true under every assignment. A satisfiable formula is true under at least one assignment, while an unsatisfiable formula is never true.

In [5]:
def classify(formula, n_vars):
    """Classify a Boolean formula by checking every possible assignment."""
    assignments = itertools.product([True, False], repeat=n_vars)
    results = [formula(*values) for values in assignments]

    if all(results):
        return "Valid (Tautology)"
    if any(results):
        return "Satisfiable (but not valid)"
    return "Unsatisfiable"

contradiction = lambda P: P and not P
excluded_middle = lambda P: P or not P
same_truth_value = lambda P, Q: (P and Q) or ((not P) and (not Q))

print(f"{'P and not P':<36} -> {classify(contradiction, 1)}")
print(f"{'P or not P':<36} -> {classify(excluded_middle, 1)}")
print(f"{'(P and Q) or (not P and not Q)':<36} -> {classify(same_truth_value, 2)}")

P and not P                         -> Unsatisfiable
P or not P                          -> Valid (Tautology)
(P and Q) or (not P and not Q)      -> Satisfiable (but not valid)


## Task 4 – Logical Equivalence and De Morgan's Laws

Two formulas are logically equivalent when they agree for every possible assignment, not merely for one example.

In [6]:
def are_equivalent(first_formula, second_formula, n_vars):
    """Return True only when both formulas agree on every assignment."""
    for values in itertools.product([True, False], repeat=n_vars):
        if first_formula(*values) != second_formula(*values):
            return False
    return True

# De Morgan's Law 2: NOT(P OR Q) is equivalent to (NOT P) AND (NOT Q).
demorgan_left = lambda P, Q: not (P or Q)
demorgan_right = lambda P, Q: (not P) and (not Q)
print("De Morgan's Law 2 is equivalent:",
      are_equivalent(demorgan_left, demorgan_right, 2))

# Let P mean 'the printer is on' and W mean 'the printer is connected to WiFi'.
printer_negated_conjunction = lambda P, W: not (P and W)
printer_demorgan_form = lambda P, W: (not P) or (not W)
print("Printer statement forms are equivalent:",
      are_equivalent(printer_negated_conjunction, printer_demorgan_form, 2))

De Morgan's Law 2 is equivalent: True
Printer statement forms are equivalent: True


## Task 5 – Checking Inference Patterns

An argument is invalid if even one assignment makes all its premises true and its conclusion false. That assignment is a counterexample.

In [7]:
def is_valid_argument(premises_func, conclusion_func, n_vars):
    """Check validity and return the first counterexample if one exists."""
    for values in itertools.product([True, False], repeat=n_vars):
        if premises_func(*values) and not conclusion_func(*values):
            return False, values
    return True, None

patterns = {
    "Modus Ponens": (lambda P, Q: impl(P, Q) and P, lambda P, Q: Q),
    "Modus Tollens": (lambda P, Q: impl(P, Q) and not Q, lambda P, Q: not P),
    "Affirming the Consequent": (lambda P, Q: impl(P, Q) and Q, lambda P, Q: P),
    "Denying the Antecedent": (lambda P, Q: impl(P, Q) and not P, lambda P, Q: not Q),
}

for name, (premises, conclusion) in patterns.items():
    valid, counterexample = is_valid_argument(premises, conclusion, 2)
    message = "Valid" if valid else f"Invalid; counterexample (P, Q) = {counterexample}"
    print(f"{name:<26} {message}")

Modus Ponens              Valid
Modus Tollens             Valid
Affirming the Consequent  Invalid; counterexample (P, Q) = (False, True)
Denying the Antecedent    Invalid; counterexample (P, Q) = (False, True)


In [8]:
# Each English argument has the same P/Q structure as one of the patterns above.
english_arguments = [
    ("1. Exam argument", "Affirming the Consequent",
     lambda P, Q: impl(P, Q) and Q, lambda P, Q: P),
    ("2. Divisibility", "Modus Tollens",
     lambda P, Q: impl(P, Q) and not Q, lambda P, Q: not P),
    ("3. Office argument", "Modus Ponens",
     lambda P, Q: impl(P, Q) and P, lambda P, Q: Q),
    ("4. Battery argument", "Denying the Antecedent",
     lambda P, Q: impl(P, Q) and not P, lambda P, Q: not Q),
]

for label, pattern, premises, conclusion in english_arguments:
    valid, counterexample = is_valid_argument(premises, conclusion, 2)
    result = "Valid" if valid else f"Invalid; counterexample = {counterexample}"
    print(f"{label:<22} | {pattern:<24} | {result}")

1. Exam argument       | Affirming the Consequent | Invalid; counterexample = (False, True)
2. Divisibility        | Modus Tollens            | Valid
3. Office argument     | Modus Ponens             | Valid
4. Battery argument    | Denying the Antecedent   | Invalid; counterexample = (False, True)


### Meaning of the variables in the English arguments

1. `P`: exam is hard; `Q`: many students fail.
2. `P`: 18 is divisible by 4; `Q`: 18 is divisible by 2.
3. `P`: today is a weekday; `Q`: the office is open.
4. `P`: battery is dead; `Q`: phone will not turn on. Therefore, ‘phone will turn on’ is `not Q`.

## Task 6 – Modelling a First-Order Logic Domain

The domain contains people and courses. Unary predicates are sets of objects, while binary predicates are sets of ordered pairs. Python's `all()` represents ∀ and `any()` represents ∃.

In [9]:
# Domain of discourse: people and courses. Constants are written as strings.
domain = {'Ali', 'Sara', 'Dr Ahmed', 'Dr Fatima', 'KRR', 'CN'}

# Unary predicates
Student = {'Ali', 'Sara'}
Professor = {'Dr Ahmed', 'Dr Fatima'}
Course = {'KRR', 'CN'}

# Binary predicates: each tuple is ordered as (person, course).
Enrolled = {('Ali', 'KRR'), ('Sara', 'CN')}
Teaches = {('Dr Ahmed', 'KRR'), ('Dr Fatima', 'CN')}
Studies = {('Ali', 'KRR'), ('Sara', 'CN')}
Fails = {('Ali', 'CN')}

# 1) Student(Ali)
# Constant: Ali. Predicate: Student. No quantifier or variable is needed.
ali_is_student = 'Ali' in Student

# 2) For every professor p, there exists a course c that p teaches.
# Quantifiers: forall p, exists c. Predicates: Professor, Course, Teaches.
every_professor_teaches_a_course = all(
    any((professor, course) in Teaches for course in Course)
    for professor in Professor
)

# 3) There exists a student s who studies the constant KRR.
# Quantifier: exists s. Predicates: Student, Studies. Constant: KRR.
some_student_studies_krr = any(
    (student, 'KRR') in Studies for student in Student
)

# 4) No student fails every course. Equivalently, every student has at least
# one course they do not fail. Quantifiers: forall s, exists c.
# Predicates: Student, Course, Fails. Variables: student, course.
no_student_fails_every_course = all(
    any((student, course) not in Fails for course in Course)
    for student in Student
)

# 5) There exists one course c such that every student s enrolled in c.
# Quantifier order matters: exists c comes before forall s.
# Predicates: Course, Student, Enrolled. Variables: course, student.
course_taken_by_every_student = any(
    all((student, course) in Enrolled for student in Student)
    for course in Course
)

print("Ali is a student:", ali_is_student)
print("Every professor teaches at least one course:", every_professor_teaches_a_course)
print("Some students study KRR:", some_student_studies_krr)
print("No student fails every course:", no_student_fails_every_course)
print("A course exists that every student enrolled in:", course_taken_by_every_student)

Ali is a student: True
Every professor teaches at least one course: True
Some students study KRR: True
No student fails every course: True
A course exists that every student enrolled in: False


### Final check

The formulas keep constants (`'Ali'`, `'KRR'`) separate from variables, preserve the intended quantifier scopes and order, and do not reverse any implication. The final FOL statement evaluates to `False` in this interpretation because Ali enrolled in KRR while Sara enrolled in CN, so there is no single course shared by every student.